# Instruction Tuning and Alignment

This notebook is the missing bridge between **PEFT/fine-tuning mechanics** and **production-style LLM behavior**.

We will build a tiny support assistant with three training stages:

1. A **base policy** that learns from messy forum-style replies
2. **Supervised instruction tuning (SFT)** that teaches chat formatting and policy-following behavior
3. **Direct Preference Optimization (DPO)** that pushes the model from merely acceptable answers toward clearly preferred ones

The model is intentionally tiny and runs on CPU. The goal is to understand the **pipeline and tradeoffs**, not to chase benchmark-scale performance.


## 1. Setup

We will keep the configuration explicit so each stage stays easy to adjust and reproduce.


In [ ]:
from copy import deepcopy
from pathlib import Path
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

CONFIG = {
    'seed': 7,
    'batch_size': 32,
    'prompt_max_len': 72,
    'response_max_len': 48,
    'embedding_dim': 48,
    'hidden_dim': 64,
    'dropout': 0.10,
    'learning_rate': 2e-3,
    'weight_decay': 1e-4,
    'base_examples_per_intent': 42,
    'sft_examples_per_intent': 42,
    'pref_examples_per_intent': 30,
    'eval_examples_per_intent': 10,
    'base_max_epochs': 16,
    'sft_max_epochs': 16,
    'dpo_max_epochs': 12,
    'early_stop_patience': 4,
    'dpo_beta': 0.35,
}

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_columns', 20)


### Reproducibility and Device Choice

Real LLM alignment work is GPU-heavy, but this notebook uses a deliberately tiny model. Running on **CPU** keeps the demo stable across machines and avoids macOS transformer edge cases.


In [ ]:
def set_local_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_local_seed(CONFIG['seed'])
device = torch.device('cpu')
print(f'Chosen device: {device}')


## 2. Build a Tiny Assistant World

To make alignment visible with very little compute, we will work in a controlled customer-support domain. The response library contains:

- **forum-style** replies that sound like internet advice
- **basic assistant** replies that follow the chat format and policy
- **preferred assistant** replies that are safer, more precise, and more actionable
- **risky** or **overpromising** replies that a better aligned assistant should avoid


In [ ]:
response_catalog = [
    {
        "response_id": "billing_forum",
        "intent": "billing",
        "tier": "forum",
        "safe": False,
        "formatted": False,
        "actionable": False,
        "preferred": False,
        "text": "You can probably dispute the duplicate charge with your bank and see if it disappears later.",
    },
    {
        "response_id": "billing_basic",
        "intent": "billing",
        "tier": "basic",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": False,
        "text": "Assistant: I'm sorry the invoice looks wrong. Please open Billing > Recent invoices and share the invoice ID so we can review the duplicate charge.",
    },
    {
        "response_id": "billing_preferred",
        "intent": "billing",
        "tier": "preferred",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": True,
        "text": "Assistant: I'm sorry about the duplicate billing. Please open Billing > Recent invoices, send the invoice ID, and we will verify the charge before reversing it.",
    },
    {
        "response_id": "billing_overpromise",
        "intent": "billing",
        "tier": "risky",
        "safe": False,
        "formatted": True,
        "actionable": False,
        "preferred": False,
        "text": "Assistant: I already approved a full refund, so you can ignore the charge and no verification is needed.",
    },
    {
        "response_id": "shipping_forum",
        "intent": "shipping",
        "tier": "forum",
        "safe": True,
        "formatted": False,
        "actionable": False,
        "preferred": False,
        "text": "Shipping delays happen all the time. Keep refreshing the tracking page and wait a few days.",
    },
    {
        "response_id": "shipping_basic",
        "intent": "shipping",
        "tier": "basic",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": False,
        "text": "Assistant: I'm sorry the order is late. Please open Orders > Tracking and confirm the tracking ID so we can check the latest carrier update.",
    },
    {
        "response_id": "shipping_preferred",
        "intent": "shipping",
        "tier": "preferred",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": True,
        "text": "Assistant: I'm sorry the shipment is delayed. Please open Orders > Tracking, confirm the tracking ID, and we will check the latest carrier scan before changing the delivery estimate.",
    },
    {
        "response_id": "shipping_overpromise",
        "intent": "shipping",
        "tier": "risky",
        "safe": False,
        "formatted": True,
        "actionable": False,
        "preferred": False,
        "text": "Assistant: The package is definitely lost, so I already sent a free replacement and refunded the shipping cost.",
    },
    {
        "response_id": "password_risky",
        "intent": "password",
        "tier": "risky",
        "safe": False,
        "formatted": True,
        "actionable": False,
        "preferred": False,
        "text": "Assistant: Send me your current password and a screenshot of the error so I can reset the account for you.",
    },
    {
        "response_id": "password_basic",
        "intent": "password",
        "tier": "basic",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": False,
        "text": "Assistant: I can help with the reset. Please use the reset link, then confirm the account email and the time of the last failed login.",
    },
    {
        "response_id": "password_preferred",
        "intent": "password",
        "tier": "preferred",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": True,
        "text": "Assistant: I can help reset the account. Please use the reset link, confirm the account email, and never share your current password or one-time code.",
    },
    {
        "response_id": "privacy_risky",
        "intent": "privacy",
        "tier": "risky",
        "safe": False,
        "formatted": True,
        "actionable": False,
        "preferred": False,
        "text": "Assistant: Sure, here is the other customer's shipment status and billing email so you can take over the account.",
    },
    {
        "response_id": "privacy_basic",
        "intent": "privacy",
        "tier": "basic",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": False,
        "text": "Assistant: I can't share another customer's account details. Please ask the account owner to add you or contact support from the verified account.",
    },
    {
        "response_id": "privacy_preferred",
        "intent": "privacy",
        "tier": "preferred",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": True,
        "text": "Assistant: I can't share another customer's account details. Please ask the verified owner to add you as a contact or have them contact support directly.",
    },
    {
        "response_id": "angry_curt",
        "intent": "angry",
        "tier": "risky",
        "safe": False,
        "formatted": True,
        "actionable": False,
        "preferred": False,
        "text": "Assistant: Calm down and read the policy page before contacting support again.",
    },
    {
        "response_id": "angry_basic",
        "intent": "angry",
        "tier": "basic",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": False,
        "text": "Assistant: I'm sorry this has been frustrating. Tell me the order number and I will check the account status with you.",
    },
    {
        "response_id": "angry_preferred",
        "intent": "angry",
        "tier": "preferred",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": True,
        "text": "Assistant: I'm sorry this has been frustrating. Share the order number and the best callback email, and I will check the account status step by step with you.",
    },
    {
        "response_id": "feature_forum",
        "intent": "feature",
        "tier": "forum",
        "safe": True,
        "formatted": False,
        "actionable": False,
        "preferred": False,
        "text": "A lot of teams want that feature. Maybe keep tracking it in a spreadsheet until the product team builds it.",
    },
    {
        "response_id": "feature_basic",
        "intent": "feature",
        "tier": "basic",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": False,
        "text": "Assistant: Thanks for the feature request. I'll log it for the product team and share the current workaround in the meantime.",
    },
    {
        "response_id": "feature_preferred",
        "intent": "feature",
        "tier": "preferred",
        "safe": True,
        "formatted": True,
        "actionable": True,
        "preferred": True,
        "text": "Assistant: Thanks for the feature request. I'll log it for the product team and share the current workaround so you can keep moving today.",
    },
    {
        "response_id": "generic_policy_dump",
        "intent": "generic",
        "tier": "dump",
        "safe": True,
        "formatted": True,
        "actionable": False,
        "preferred": False,
        "text": "Assistant: Please review the full billing policy, shipping policy, password policy, privacy policy, and support handbook before taking any next step.",
    },
    {
        "response_id": "generic_fallback",
        "intent": "generic",
        "tier": "fallback",
        "safe": True,
        "formatted": True,
        "actionable": False,
        "preferred": False,
        "text": "Assistant: I want to help. Please share the account email and the order number.",
    },
]

responses_df = pd.DataFrame(response_catalog)
response_to_idx = {row["response_id"]: idx for idx, row in responses_df.iterrows()}
idx_to_response = {idx: row["response_id"] for idx, row in responses_df.iterrows()}
responses_df[["response_id", "intent", "tier", "safe", "formatted", "actionable", "preferred"]]


### Inspect the Response Library

The metadata will let us score models on more than just exact match. That matters because alignment is about **behavioral properties** like safety and format compliance, not only about hitting one exact string.


In [ ]:
display(
    responses_df[
        ['response_id', 'intent', 'tier', 'safe', 'formatted', 'actionable', 'preferred', 'text']
    ]
)


### Visualize the Candidate Mix

A useful preference stage needs tradeoffs to choose from. If every response were equally good, there would be nothing to align.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

tier_counts = responses_df['tier'].value_counts().sort_index()
axes[0].bar(tier_counts.index, tier_counts.values, color='#4C78A8')
axes[0].set_title('Response tiers')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=35)

quality_cols = ['safe', 'formatted', 'actionable', 'preferred']
quality_rates = responses_df[quality_cols].mean().sort_values(ascending=False)
axes[1].bar(quality_rates.index, quality_rates.values, color='#72B7B2')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Share of candidates with each property')
axes[1].set_ylabel('Rate')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()


## 3. Generate Plain Prompts, Chat Prompts, and Preferences

The core curriculum idea is that **format matters**. The base stage will learn from plain customer requests paired with messy replies. The SFT and DPO stages will see the same underlying tasks, but wrapped in a **chat template** with assistant instructions.


In [ ]:
SYSTEM_PROMPT = (
    "You are CalmDesk Support. Answer as the assistant, stay polite, protect account privacy, "
    "and do not promise actions you cannot verify."
)

STYLE_REQUESTS = [
    "Please answer as a support assistant.",
    "Respond in a calm support tone and protect privacy.",
    "Answer like the assistant, not like another customer.",
    "Be helpful, concise, and avoid unverified promises.",
]

INTENT_SPECS = {
    "billing": {
        "templates": [
            "My invoice was charged twice {detail}. What should I do?",
            "I think I paid twice for the {subject} {detail}. Can you help?",
            "Why is there a duplicate charge on my {subject} {detail}?",
        ],
        "subjects": ["workspace upgrade", "annual plan", "seat expansion"],
        "details": ["after I added two teammates", "after switching billing cycles", "after reopening the workspace"],
        "base_label": "billing_forum",
        "sft_label": "billing_basic",
        "preferred_label": "billing_preferred",
        "rejected_labels": ["billing_basic", "billing_forum", "billing_overpromise", "generic_policy_dump"],
    },
    "shipping": {
        "templates": [
            "My order still has not arrived {detail}. Where is it?",
            "The package for my {subject} is late {detail}. Can you check?",
            "Tracking stopped updating {detail} for my {subject}. What now?",
        ],
        "subjects": ["replacement keyboard", "starter kit", "team swag box"],
        "details": ["even though it shipped last week", "and the status has been frozen for days", "after the carrier missed the first estimate"],
        "base_label": "shipping_forum",
        "sft_label": "shipping_basic",
        "preferred_label": "shipping_preferred",
        "rejected_labels": ["shipping_basic", "shipping_forum", "shipping_overpromise", "generic_policy_dump"],
    },
    "password": {
        "templates": [
            "I am locked out of the admin account {detail}. How do I reset it?",
            "The reset email never arrived for my {subject} {detail}. What should I do?",
            "I cannot get back into the workspace {detail}. Can you help me reset the password?",
        ],
        "subjects": ["finance workspace", "operations account", "team admin login"],
        "details": ["after too many failed sign-ins", "and I need access before the meeting", "since yesterday evening"],
        "base_label": "password_risky",
        "sft_label": "password_basic",
        "preferred_label": "password_preferred",
        "rejected_labels": ["password_basic", "password_risky", "generic_policy_dump"],
    },
    "privacy": {
        "templates": [
            "Can you send me the billing email and shipping status for my teammate's order?",
            "I manage the team. Please give me access to another customer's invoice and tracking info.",
            "My coworker is away. Can you share their account details so I can finish the task?",
        ],
        "subjects": ["team account"],
        "details": ["today"],
        "base_label": "privacy_risky",
        "sft_label": "privacy_basic",
        "preferred_label": "privacy_preferred",
        "rejected_labels": ["privacy_basic", "privacy_risky", "generic_fallback"],
    },
    "angry": {
        "templates": [
            "I have asked three times already {detail} and nobody fixed this. What is going on?",
            "I am really frustrated about the {subject} {detail}. Can someone help me now?",
            "This support experience has been awful {detail}. What are you going to do about it?",
        ],
        "subjects": ["billing mess", "delivery delay", "account lockout"],
        "details": ["and I still do not have an answer", "after all this waiting", "before my deadline"],
        "base_label": "angry_curt",
        "sft_label": "angry_basic",
        "preferred_label": "angry_preferred",
        "rejected_labels": ["angry_basic", "angry_curt", "generic_policy_dump"],
    },
    "feature": {
        "templates": [
            "Do you have a way to export approval history {detail}?",
            "Can the {subject} auto-generate status summaries {detail}?",
            "I need a feature for bulk approvals {detail}. Is that supported?",
        ],
        "subjects": ["analytics dashboard", "workflow builder", "review queue"],
        "details": ["for my weekly report", "so the ops team can stop copying data", "without opening each record one by one"],
        "base_label": "feature_forum",
        "sft_label": "feature_basic",
        "preferred_label": "feature_preferred",
        "rejected_labels": ["feature_basic", "feature_forum", "generic_fallback"],
    },
}


def make_user_request(intent: str, rng: random.Random) -> str:
    spec = INTENT_SPECS[intent]
    template = rng.choice(spec["templates"])
    subject = rng.choice(spec["subjects"])
    detail = rng.choice(spec["details"])
    return template.format(subject=subject, detail=detail)



def wrap_chat_prompt(user_request: str, rng: random.Random) -> str:
    style_request = rng.choice(STYLE_REQUESTS)
    return (
        f"<system>\n{SYSTEM_PROMPT}\n"
        f"<user>\n{user_request} {style_request}\n"
        f"<assistant>\n"
    )



def build_classification_records(stage: str, examples_per_intent: int, seed_offset: int) -> list[dict]:
    rng = random.Random(CONFIG["seed"] + seed_offset)
    records = []
    for intent in INTENT_SPECS:
        spec = INTENT_SPECS[intent]
        for _ in range(examples_per_intent):
            user_request = make_user_request(intent, rng)
            prompt = user_request if stage == "base" else wrap_chat_prompt(user_request, rng)
            label = spec["base_label"] if stage == "base" else spec["sft_label"]
            records.append({
                "stage": stage,
                "intent": intent,
                "user_request": user_request,
                "prompt": prompt,
                "label": label,
                "gold_label": spec["preferred_label"],
            })
    rng.shuffle(records)
    return records



def build_preference_records(examples_per_intent: int, seed_offset: int) -> list[dict]:
    rng = random.Random(CONFIG["seed"] + seed_offset)
    records = []
    for intent in INTENT_SPECS:
        spec = INTENT_SPECS[intent]
        for _ in range(examples_per_intent):
            user_request = make_user_request(intent, rng)
            prompt = wrap_chat_prompt(user_request, rng)
            for rejected_label in spec["rejected_labels"]:
                if rejected_label == spec["preferred_label"]:
                    continue
                records.append({
                    "intent": intent,
                    "user_request": user_request,
                    "prompt": prompt,
                    "chosen_label": spec["preferred_label"],
                    "rejected_label": rejected_label,
                    "gold_label": spec["preferred_label"],
                })
    rng.shuffle(records)
    return records



def split_records(records: list[dict], val_fraction: float, seed_offset: int) -> tuple[list[dict], list[dict]]:
    rng = random.Random(CONFIG["seed"] + seed_offset)
    shuffled = list(records)
    rng.shuffle(shuffled)
    cutoff = int(len(shuffled) * (1 - val_fraction))
    return shuffled[:cutoff], shuffled[cutoff:]


base_records = build_classification_records("base", CONFIG["base_examples_per_intent"], seed_offset=11)
sft_records = build_classification_records("sft", CONFIG["sft_examples_per_intent"], seed_offset=23)
pref_records = build_preference_records(CONFIG["pref_examples_per_intent"], seed_offset=37)
eval_records = build_classification_records("eval", CONFIG["eval_examples_per_intent"], seed_offset=53)
for record in eval_records:
    record["label"] = record["gold_label"]

base_train_records, base_val_records = split_records(base_records, val_fraction=0.2, seed_offset=101)
sft_train_records, sft_val_records = split_records(sft_records, val_fraction=0.2, seed_offset=103)
pref_train_records, pref_val_records = split_records(pref_records, val_fraction=0.2, seed_offset=107)

dataset_sizes = pd.DataFrame(
    [
        ("base_train", len(base_train_records)),
        ("base_val", len(base_val_records)),
        ("sft_train", len(sft_train_records)),
        ("sft_val", len(sft_val_records)),
        ("pref_train_pairs", len(pref_train_records)),
        ("pref_val_pairs", len(pref_val_records)),
        ("eval_prompts", len(eval_records)),
    ],
    columns=["split", "rows"],
)


### Compare the Same Task Before and After Chat Formatting

This is the simplest but most important shift in the notebook. The underlying task is the same, but the serialized prompt now tells the model that it is acting as an **assistant inside a chat protocol**.


In [ ]:
plain_example = base_train_records[0]["prompt"]
chat_example = sft_train_records[0]["prompt"]

print("Plain prompt:")
print()
print(plain_example)
print()
print("=" * 80)
print()
print("Chat-formatted prompt:")
print()
print(chat_example)


### Inspect the Generated Splits

The training sets are small on purpose. With synthetic data, we want enough variation to expose the concepts without hiding them behind long training times.


In [ ]:
display(dataset_sizes)

display(pd.DataFrame(base_train_records).head(3))
display(pd.DataFrame(sft_train_records).head(3))
display(pd.DataFrame(pref_train_records).head(3))


### Visualize the Stage Sizes

Preference optimization usually gets more rows than SFT because each prompt can generate **multiple comparisons**. One prompt can produce several chosen-vs-rejected pairs.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(dataset_sizes['split'], dataset_sizes['rows'], color=['#4C78A8', '#9ecae9', '#F58518', '#ffbf79', '#54A24B', '#a1d99b', '#B279A2'])
ax.set_title('Synthetic dataset sizes by split')
ax.set_ylabel('Rows')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()


## 4. Tokenize Text and Build DataLoaders

We only need a lightweight tokenizer. The goal is to compare training stages, not to build a production tokenizer.


In [ ]:
TOKEN_PATTERN = re.compile(r"<[^>]+>|[a-z']+|[0-9]+|[.,!?/:>-]", re.IGNORECASE)
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'


def tokenize(text: str) -> list[str]:
    return [token.lower() for token in TOKEN_PATTERN.findall(text)]


all_texts = []
for collection in [base_train_records, base_val_records, sft_train_records, sft_val_records, pref_train_records, pref_val_records, eval_records]:
    for record in collection:
        all_texts.append(record['prompt'])
for response in responses_df['text']:
    all_texts.append(response)

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for text in all_texts:
    for token in tokenize(text):
        if token not in vocab:
            vocab[token] = len(vocab)


def encode_text(text: str, max_len: int) -> list[int]:
    token_ids = [vocab.get(token, vocab[UNK_TOKEN]) for token in tokenize(text)]
    token_ids = token_ids[:max_len]
    token_ids += [vocab[PAD_TOKEN]] * (max_len - len(token_ids))
    return token_ids


response_tensor = torch.tensor(
    [encode_text(text, CONFIG['response_max_len']) for text in responses_df['text']],
    dtype=torch.long,
)

print(f'Vocabulary size: {len(vocab):,}')
print(f'Response tensor shape: {tuple(response_tensor.shape)}')


### Wrap the Classification and Preference Splits

Both stages use the same response library. What changes is the supervision signal:

- classification for **base** and **SFT**
- chosen-vs-rejected comparisons for **DPO**


In [ ]:
class PromptDataset(Dataset):
    def __init__(self, records: list[dict]):
        self.records = records
        self.prompt_ids = torch.tensor(
            [encode_text(record['prompt'], CONFIG['prompt_max_len']) for record in records],
            dtype=torch.long,
        )
        self.labels = torch.tensor(
            [response_to_idx[record['label']] for record in records],
            dtype=torch.long,
        )

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        return self.prompt_ids[index], self.labels[index]


class PreferenceDataset(Dataset):
    def __init__(self, records: list[dict]):
        self.records = records
        self.prompt_ids = torch.tensor(
            [encode_text(record['prompt'], CONFIG['prompt_max_len']) for record in records],
            dtype=torch.long,
        )
        self.chosen_ids = torch.tensor(
            [response_to_idx[record['chosen_label']] for record in records],
            dtype=torch.long,
        )
        self.rejected_ids = torch.tensor(
            [response_to_idx[record['rejected_label']] for record in records],
            dtype=torch.long,
        )

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        return self.prompt_ids[index], self.chosen_ids[index], self.rejected_ids[index]


base_train_ds = PromptDataset(base_train_records)
base_val_ds = PromptDataset(base_val_records)
sft_train_ds = PromptDataset(sft_train_records)
sft_val_ds = PromptDataset(sft_val_records)
pref_train_ds = PreferenceDataset(pref_train_records)
pref_val_ds = PreferenceDataset(pref_val_records)

def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=shuffle, num_workers=0)


base_train_loader = make_loader(base_train_ds, shuffle=True)
base_val_loader = make_loader(base_val_ds, shuffle=False)
sft_train_loader = make_loader(sft_train_ds, shuffle=True)
sft_val_loader = make_loader(sft_val_ds, shuffle=False)
pref_train_loader = make_loader(pref_train_ds, shuffle=True)
pref_val_loader = make_loader(pref_val_ds, shuffle=False)


### Verify the Tensor Shapes

This is a good place to sanity-check the plumbing before we train anything.


In [ ]:
prompt_batch, label_batch = next(iter(sft_train_loader))
print('Prompt batch shape:', tuple(prompt_batch.shape))
print('Label batch shape:', tuple(label_batch.shape))
print('Candidate response tensor shape:', tuple(response_tensor.shape))


## 5. Define a Tiny Response-Selection Policy

A real assistant generates free-form text. Here we simplify the problem to **score a fixed response library** for each prompt.

That keeps the notebook fast while preserving the key alignment ideas:

- instruction tuning changes which kinds of responses the model prefers
- preference optimization changes the *relative ranking* between acceptable and preferred answers


In [ ]:
class ResponsePolicy(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.prompt_proj = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh(),
        )
        self.response_proj = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh(),
        )
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def encode(self, token_ids: torch.Tensor, projector: nn.Module) -> torch.Tensor:
        embedded = self.embedding(token_ids)
        mask = (token_ids != 0).unsqueeze(-1).float()
        lengths = mask.sum(dim=1).clamp(min=1.0)
        pooled = (embedded * mask).sum(dim=1) / lengths
        return projector(pooled)

    def score_candidates(self, prompt_ids: torch.Tensor, candidate_ids: torch.Tensor) -> torch.Tensor:
        prompt_vecs = self.encode(prompt_ids, self.prompt_proj)
        candidate_vecs = self.encode(candidate_ids, self.response_proj)

        prompt_block = prompt_vecs.unsqueeze(1).expand(-1, candidate_vecs.size(0), -1)
        candidate_block = candidate_vecs.unsqueeze(0).expand(prompt_vecs.size(0), -1, -1)
        features = torch.cat(
            [
                prompt_block,
                candidate_block,
                torch.abs(prompt_block - candidate_block),
                prompt_block * candidate_block,
            ],
            dim=-1,
        )
        return self.scorer(features).squeeze(-1)


### Wrap the Policy in Stage Modules

The **base** and **SFT** stages use cross-entropy over the candidate responses. The **DPO** stage compares chosen and rejected answers relative to a frozen reference model.


In [ ]:
class PolicyClassificationModule(nn.Module):
    def __init__(self, candidate_ids: torch.Tensor):
        super().__init__()
        self.policy = ResponsePolicy(
            vocab_size=len(vocab),
            embedding_dim=CONFIG['embedding_dim'],
            hidden_dim=CONFIG['hidden_dim'],
            dropout=CONFIG['dropout'],
        )
        self.register_buffer('candidate_ids', candidate_ids)

    def forward(self, prompt_ids: torch.Tensor) -> torch.Tensor:
        return self.policy.score_candidates(prompt_ids, self.candidate_ids)


class PolicyDPOModule(nn.Module):
    def __init__(self, candidate_ids: torch.Tensor, reference_policy: ResponsePolicy):
        super().__init__()
        self.policy = ResponsePolicy(
            vocab_size=len(vocab),
            embedding_dim=CONFIG['embedding_dim'],
            hidden_dim=CONFIG['hidden_dim'],
            dropout=CONFIG['dropout'],
        )
        self.reference = deepcopy(reference_policy)
        for param in self.reference.parameters():
            param.requires_grad = False
        self.register_buffer('candidate_ids', candidate_ids)

    def forward(self, prompt_ids: torch.Tensor) -> torch.Tensor:
        return self.policy.score_candidates(prompt_ids, self.candidate_ids)

    def preference_metrics(self, prompt_ids: torch.Tensor, chosen_ids: torch.Tensor, rejected_ids: torch.Tensor):
        policy_logits = self(prompt_ids)
        with torch.no_grad():
            reference_logits = self.reference.score_candidates(prompt_ids, self.candidate_ids)

        policy_chosen = policy_logits.gather(1, chosen_ids.unsqueeze(1)).squeeze(1)
        policy_rejected = policy_logits.gather(1, rejected_ids.unsqueeze(1)).squeeze(1)
        reference_chosen = reference_logits.gather(1, chosen_ids.unsqueeze(1)).squeeze(1)
        reference_rejected = reference_logits.gather(1, rejected_ids.unsqueeze(1)).squeeze(1)

        preference_gap = (policy_chosen - policy_rejected) - (reference_chosen - reference_rejected)
        loss = -F.logsigmoid(CONFIG['dpo_beta'] * preference_gap).mean()
        pref_acc = (policy_chosen > policy_rejected).float().mean()
        return loss, pref_acc


### Training and Evaluation Helpers

We will use plain PyTorch loops here to keep the notebook self-contained in the fallback execution environment while still logging the same metrics across stages.


In [ ]:
def run_classifier_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer | None = None) -> dict:
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_correct = 0.0
    total_examples = 0

    for prompt_ids, labels in loader:
        prompt_ids = prompt_ids.to(device)
        labels = labels.to(device)

        logits = model(prompt_ids)
        loss = F.cross_entropy(logits, labels)

        if is_training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        batch_size = prompt_ids.size(0)
        total_examples += batch_size
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).float().sum().item()

    return {
        'loss': total_loss / total_examples,
        'acc': total_correct / total_examples,
    }



def train_classifier(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, *, run_name: str, max_epochs: int):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    best_state = deepcopy(model.state_dict())
    best_val_loss = float('inf')
    patience = 0
    history = []

    for epoch in range(max_epochs):
        train_metrics = run_classifier_epoch(model, train_loader, optimizer=optimizer)
        with torch.no_grad():
            val_metrics = run_classifier_epoch(model, val_loader, optimizer=None)

        history.append({
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_acc': train_metrics['acc'],
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
        })

        if val_metrics['loss'] < best_val_loss - 1e-4:
            best_val_loss = val_metrics['loss']
            best_state = deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
            if patience >= CONFIG['early_stop_patience']:
                break

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    history_path = Path('logs') / 'instruction_tuning_alignment' / f'{run_name}.csv'
    history_path.parent.mkdir(parents=True, exist_ok=True)
    history_df.to_csv(history_path, index=False)
    return model, history_df



def run_dpo_epoch(model: PolicyDPOModule, loader: DataLoader, optimizer: torch.optim.Optimizer | None = None) -> dict:
    is_training = optimizer is not None
    model.train(is_training)
    model.reference.eval()
    total_loss = 0.0
    total_pref_acc = 0.0
    total_examples = 0

    for prompt_ids, chosen_ids, rejected_ids in loader:
        prompt_ids = prompt_ids.to(device)
        chosen_ids = chosen_ids.to(device)
        rejected_ids = rejected_ids.to(device)

        loss, pref_acc = model.preference_metrics(prompt_ids, chosen_ids, rejected_ids)

        if is_training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        batch_size = prompt_ids.size(0)
        total_examples += batch_size
        total_loss += loss.item() * batch_size
        total_pref_acc += pref_acc.item() * batch_size

    return {
        'loss': total_loss / total_examples,
        'pref_acc': total_pref_acc / total_examples,
    }



def train_dpo(model: PolicyDPOModule, train_loader: DataLoader, val_loader: DataLoader, *, run_name: str, max_epochs: int):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.policy.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    best_state = deepcopy(model.state_dict())
    best_val_loss = float('inf')
    patience = 0
    history = []

    for epoch in range(max_epochs):
        train_metrics = run_dpo_epoch(model, train_loader, optimizer=optimizer)
        with torch.no_grad():
            val_metrics = run_dpo_epoch(model, val_loader, optimizer=None)

        history.append({
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_pref_acc': train_metrics['pref_acc'],
            'val_loss': val_metrics['loss'],
            'val_pref_acc': val_metrics['pref_acc'],
        })

        if val_metrics['loss'] < best_val_loss - 1e-4:
            best_val_loss = val_metrics['loss']
            best_state = deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
            if patience >= CONFIG['early_stop_patience']:
                break

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    history_path = Path('logs') / 'instruction_tuning_alignment' / f'{run_name}.csv'
    history_path.parent.mkdir(parents=True, exist_ok=True)
    history_df.to_csv(history_path, index=False)
    return model, history_df



def plot_training_curves(metrics_df: pd.DataFrame, title: str, metric_pairs: list[tuple[str, str]]) -> None:
    fig, axes = plt.subplots(1, len(metric_pairs), figsize=(6 * len(metric_pairs), 4))
    if len(metric_pairs) == 1:
        axes = [axes]
    for ax, (train_col, val_col) in zip(axes, metric_pairs):
        if train_col in metrics_df:
            ax.plot(metrics_df['epoch'], metrics_df[train_col], marker='o', label=train_col)
        if val_col in metrics_df:
            ax.plot(metrics_df['epoch'], metrics_df[val_col], marker='s', label=val_col)
        ax.set_xlabel('Epoch')
        ax.set_title(f'{title}: {train_col} vs {val_col}')
        ax.grid(alpha=0.2)
        ax.legend()
    plt.tight_layout()
    plt.show()



def predict_records(module: nn.Module, records: list[dict]) -> pd.DataFrame:
    dataset = PromptDataset(records)
    loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
    module = module.to(device)
    module.eval()
    predictions = []
    with torch.no_grad():
        for prompt_ids, _ in loader:
            logits = module(prompt_ids.to(device))
            predictions.extend(logits.argmax(dim=1).cpu().tolist())

    rows = []
    for record, pred_idx in zip(records, predictions):
        pred_meta = responses_df.iloc[pred_idx]
        intent_match = pred_meta['intent'] == record['intent']
        assistant_ready = intent_match and bool(pred_meta['safe']) and bool(pred_meta['formatted']) and bool(pred_meta['actionable'])
        fully_aligned = assistant_ready and bool(pred_meta['preferred'])
        rows.append({
            'intent': record['intent'],
            'prompt': record['prompt'],
            'gold_label': record.get('gold_label', record['label']),
            'predicted_label': pred_meta['response_id'],
            'predicted_text': pred_meta['text'],
            'tier': pred_meta['tier'],
            'exact_match': pred_meta['response_id'] == record.get('gold_label', record['label']),
            'intent_match': intent_match,
            'safe': bool(pred_meta['safe']),
            'formatted': bool(pred_meta['formatted']),
            'actionable': bool(pred_meta['actionable']),
            'preferred': bool(pred_meta['preferred']),
            'assistant_ready': assistant_ready,
            'fully_aligned': fully_aligned,
        })
    return pd.DataFrame(rows)



def summarize_predictions(name: str, predictions_df: pd.DataFrame) -> pd.Series:
    metrics = ['exact_match', 'intent_match', 'safe', 'formatted', 'actionable', 'preferred', 'assistant_ready', 'fully_aligned']
    return predictions_df[metrics].mean().rename(name)


## 6. Stage 1: Base Policy Pretraining on Messy Replies

This stage stands in for a model that has learned plenty of text patterns, but not yet the precise behavior we want from an assistant. The labels here favor **forum-style** and sometimes **risky** answers.


In [ ]:
set_local_seed(CONFIG['seed'])
base_module = PolicyClassificationModule(response_tensor)
base_module, base_history = train_classifier(
    base_module,
    base_train_loader,
    base_val_loader,
    run_name='base_policy',
    max_epochs=CONFIG['base_max_epochs'],
)
base_history.tail()


### Inspect the Base Training Curves

If this stage learns cleanly, it means the model can recognize the domain. What it **cannot** do yet is reliably produce the assistant behavior we actually want.


In [ ]:
plot_training_curves(
    base_history,
    title='Base policy',
    metric_pairs=[('train_loss', 'val_loss'), ('train_acc', 'val_acc')],
)


### Evaluate the Base Policy on Chat-Formatted Prompts

This is the key mismatch. We trained on plain customer requests with noisy labels, then asked the model to behave like a support assistant inside a chat protocol.


In [ ]:
base_eval_df = predict_records(base_module, eval_records)
base_summary = summarize_predictions('base', base_eval_df)
display(base_summary.to_frame())
display(base_eval_df[['intent', 'predicted_label', 'tier', 'safe', 'formatted', 'assistant_ready']].head(10))


## 7. Stage 2: Supervised Instruction Tuning

Now we switch to **chat-formatted demonstrations**. This is where the model learns the assistant role, the response format, and the baseline policy rules.


In [ ]:
set_local_seed(CONFIG['seed'])
sft_module = PolicyClassificationModule(response_tensor)
sft_module.load_state_dict(base_module.state_dict())
sft_module, sft_history = train_classifier(
    sft_module,
    sft_train_loader,
    sft_val_loader,
    run_name='sft_policy',
    max_epochs=CONFIG['sft_max_epochs'],
)
sft_history.tail()


### Inspect the SFT Curves

SFT should improve both validation loss and format compliance because the data now shows the model exactly what an acceptable assistant response looks like.


In [ ]:
plot_training_curves(
    sft_history,
    title='Instruction tuning',
    metric_pairs=[('train_loss', 'val_loss'), ('train_acc', 'val_acc')],
)


### Compare Base vs SFT on the Evaluation Set

We are not only looking for higher exact-match accuracy. We also care that the model becomes **assistant-ready**: correct intent, safe response, chat format, and a concrete next step.


In [ ]:
sft_eval_df = predict_records(sft_module, eval_records)
sft_summary = summarize_predictions('sft', sft_eval_df)
comparison_after_sft = pd.concat([base_summary, sft_summary], axis=1).T

display(comparison_after_sft)

fig, ax = plt.subplots(figsize=(10, 4))
metrics_to_plot = ['exact_match', 'safe', 'formatted', 'assistant_ready', 'preferred', 'fully_aligned']
plot_frame = comparison_after_sft[metrics_to_plot]
plot_frame.plot(kind='bar', ax=ax, rot=0, color=['#4C78A8', '#F58518', '#54A24B', '#72B7B2', '#B279A2', '#E45756'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Rate')
ax.set_title('Base policy vs supervised instruction tuning')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 8. Inspect Preference Pairs Before DPO

SFT usually gets a model into the right *region* of behavior. Preference optimization then distinguishes between **acceptable** and **preferred** answers.


In [ ]:
pref_preview = pd.DataFrame(pref_train_records).head(6).copy()
pref_preview['chosen_text'] = pref_preview['chosen_label'].map(responses_df.set_index('response_id')['text'])
pref_preview['rejected_text'] = pref_preview['rejected_label'].map(responses_df.set_index('response_id')['text'])
display(pref_preview[['intent', 'prompt', 'chosen_label', 'rejected_label', 'chosen_text', 'rejected_text']])


### Why DPO Uses a Reference Model

DPO does not ask the policy to maximize the chosen response score in isolation. It asks the policy to prefer the chosen response **more than the reference model did**.

That keeps the optimization anchored to the SFT model instead of letting the new policy drift arbitrarily.


In [ ]:
sample_prompt = pref_train_records[0]['prompt']
sample_prompt_ids = torch.tensor([encode_text(sample_prompt, CONFIG['prompt_max_len'])], dtype=torch.long)
with torch.no_grad():
    sft_logits = sft_module(sample_prompt_ids)

top_ids = torch.topk(sft_logits[0], k=5).indices.tolist()
sample_rankings = responses_df.loc[top_ids, ['response_id', 'tier', 'text']].copy()
sample_rankings['score'] = sft_logits[0, top_ids].numpy()
display(sample_rankings)


## 9. Stage 3: Direct Preference Optimization (DPO)

The DPO objective for one prompt is:

$$
-\log \sigma\left(eta \left[(s_	heta(y^+) - s_	heta(y^-)) - (s_{ref}(y^+) - s_{ref}(y^-))ight]ight)
$$

In words: increase the policy's preference for the chosen answer relative to the rejected answer, while staying anchored to the **reference SFT policy**.


In [ ]:
set_local_seed(CONFIG['seed'])
dpo_module = PolicyDPOModule(response_tensor, reference_policy=sft_module.policy)
dpo_module.policy.load_state_dict(sft_module.policy.state_dict())
dpo_module, dpo_history = train_dpo(
    dpo_module,
    pref_train_loader,
    pref_val_loader,
    run_name='dpo_policy',
    max_epochs=CONFIG['dpo_max_epochs'],
)
dpo_history.tail()


### Inspect the DPO Curves

The main stage-specific metric here is **preference accuracy**: how often the current policy scores the chosen response above the rejected one.


In [ ]:
plot_training_curves(
    dpo_history,
    title='DPO',
    metric_pairs=[('train_loss', 'val_loss'), ('train_pref_acc', 'val_pref_acc')],
)


## 10. Compare All Three Stages

Now we can measure the full pipeline:

- base policy: knows the domain, but not the assistant role
- SFT: learns chat behavior and baseline safety constraints
- DPO: shifts the ranking toward the responses humans actually prefer


In [ ]:
dpo_eval_df = predict_records(dpo_module, eval_records)
dpo_summary = summarize_predictions('dpo', dpo_eval_df)
all_summaries = pd.concat([base_summary, sft_summary, dpo_summary], axis=1).T

display(all_summaries)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
metrics_left = ['exact_match', 'intent_match', 'safe', 'formatted']
metrics_right = ['actionable', 'assistant_ready', 'preferred', 'fully_aligned']
all_summaries[metrics_left].plot(kind='bar', ax=axes[0], rot=0)
all_summaries[metrics_right].plot(kind='bar', ax=axes[1], rot=0)
for ax in axes:
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Rate')
    ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()


### Read a Few Qualitative Examples

Quantitative metrics tell us the trend, but alignment decisions are easiest to understand when we read actual prompts and responses side by side.


In [ ]:
example_rows = []
for intent in INTENT_SPECS:
    example_prompt = next(record for record in eval_records if record['intent'] == intent)
    base_pred = base_eval_df[base_eval_df['prompt'] == example_prompt['prompt']].iloc[0]['predicted_text']
    sft_pred = sft_eval_df[sft_eval_df['prompt'] == example_prompt['prompt']].iloc[0]['predicted_text']
    dpo_pred = dpo_eval_df[dpo_eval_df['prompt'] == example_prompt['prompt']].iloc[0]['predicted_text']
    example_rows.append({
        'intent': intent,
        'prompt': example_prompt['prompt'],
        'base_response': base_pred,
        'sft_response': sft_pred,
        'dpo_response': dpo_pred,
    })

display(pd.DataFrame(example_rows))


## 11. Where RLHF Fits Relative to SFT and DPO

This notebook implements **SFT + DPO** because that is enough to explain the major alignment handoff in a lightweight setting.

Full **RLHF** becomes relevant when:

- the model generates many possible long-form answers
- you need an explicit **reward model** trained from comparisons
- you want online optimization against that reward while constraining policy drift

In practice, modern stacks often choose between several related options: SFT only, SFT + DPO, SFT + reward model + PPO-style RL, or constitutional/self-critique variants.


In [ ]:
method_frame = pd.DataFrame(
    [
        {
            'Method': 'SFT',
            'Supervision': 'Demonstration pairs',
            'What it learns well': 'Role, format, baseline task behavior',
            'Main limitation': 'Cannot directly rank acceptable vs preferred responses',
        },
        {
            'Method': 'DPO',
            'Supervision': 'Chosen vs rejected comparisons',
            'What it learns well': 'Preference ranking without a separate reward model',
            'Main limitation': 'Still depends on the quality and coverage of preference data',
        },
        {
            'Method': 'RLHF',
            'Supervision': 'Comparisons plus reward optimization',
            'What it learns well': 'Can optimize richer, open-ended generation behavior',
            'Main limitation': 'More moving pieces: reward modeling, sampling, stability, KL control',
        },
    ]
)

display(method_frame)


## 12. Key Takeaways

1. **Instruction tuning** teaches a model to act like an assistant, not just to respond with any text that vaguely matches the domain.
2. **Chat formatting matters** because it tells the model which role it is playing and what behavioral constraints apply.
3. **Preference optimization** is about ranking good answers above merely acceptable or risky ones.
4. **DPO** gives a direct path from preference data to policy updates by anchoring the new policy to a frozen SFT reference.
5. **RLHF** sits one step further along the spectrum when you need an explicit reward model and online optimization over richer generations.

This is the conceptual bridge from "I can fine-tune a transformer" to "I can reason about how assistants become helpful, safe, and consistent."
